# AirSense: Exploratory Data Analysis & Data Preprocessing
### **College Project - 3rd Year CSE**
**Project:** Air Quality Forecasting and Health Alert Platform for Indian Cities  
**Dataset:** Kaggle "Air Quality Data in India" (`city_day.csv`)  
**Focus Cities:** Delhi, Mumbai, Kolkata, Chennai, Bengaluru, Hyderabad, Ahmedabad, Jaipur  

---
### **Key Goals of this Notebook:**
1. **Load and filter** data for the 8 target Indian metropolitan hubs.
2. **Handle missing values explicitly** using time-series aware interpolation without data leakage.
3. **Detect and treat outliers** (negative sensor glitches and uncalibrated readings) aligned with Central Pollution Control Board (CPCB) NAQI regulatory caps.
4. **Analyze historical trends and seasonality** (e.g., winter inversion smog vs. monsoon rain washout, COVID-19 lockdown impact).
5. **Examine pollutant correlations** driving final AQI (PM2.5, PM10, NO2, CO, etc.).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 6)

from src.data_loader import load_raw_city_day, filter_target_cities, TARGET_CITIES, get_data_summary
from src.preprocessing import clean_and_preprocess_dataset, save_cleaned_data, CPCB_BUCKETS, assign_cpcb_bucket

## 1. Data Ingestion & Initial Inspection
We load the raw `city_day.csv` and filter it strictly for our 8 target Indian cities.

In [ ]:
raw_df = load_raw_city_day()
df_target = filter_target_cities(raw_df)

print("Target cities filtered shape:", df_target.shape)
print("\nDataset Summary:")
for k, v in get_data_summary(df_target).items():
    print(f"  {k}: {v}")

df_target.head()

## 2. Missing Value Profiling & Handling Strategy

### **Why are values missing?**
Air quality monitoring stations in India experience power cuts, routine sensor calibration, and optical lens contamination. In addition, some cities (like Mumbai and Kolkata) did not have continuous online monitoring until 2018, leading to extensive leading null records.

### **Methodology:**
1. **Pre-monitoring Trimming:** We trim dates prior to the initial commissioning date for each city. For example, backfilling 3 years of non-existent sensor data for Mumbai prior to May 2018 creates artificial flatlines.
2. **Time-Weighted Linear Interpolation:** Short sensor dropouts (1 to 7 days) are interpolated temporally along the timeline. This respects chronological continuity, unlike global mean imputation.
3. **Forward/Backward Fill (`ffill` / `bfill`):** Fills edge boundaries per city.
4. **City Median Fallback:** Used only for completely unmeasured trace gases (e.g. Xylene).

In [ ]:
missing_pct = (df_target.isna().mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_pct.values, y=missing_pct.index, hue=missing_pct.index, palette="Reds_r", legend=False)
plt.title("Missing Value Percentage by Column in Target Cities", fontsize=13, fontweight="bold")
plt.xlabel("Percentage Missing (%)")
plt.show()

## 3. Outlier Analysis & Regulatory Domain Capping

### **Why not use standard 3-sigma / Z-score outlier removal?**
Air pollution in India exhibits genuine extreme events (e.g., Diwali night firecrackers, post-monsoon stubble burning in Punjab/Haryana impacting Delhi, sandstorms in Rajasthan). Dropping high AQI rows blinds our forecasting models to dangerous pollution spikes.

### **CPCB NAQI Regulatory Capping:**
- Under the Central Pollution Control Board (CPCB) NAQI guidelines, the official scale runs from **0 to 500** (with 401-500 classified as 'Severe').
- In raw data, uncalibrated sensor drift produced readings exceeding 2000 in Ahmedabad. We cap AQI at **500** (Winsorization). This prevents extreme gradient explosion during gradient-boosting model training while preserving true hazard severity.

In [ ]:
print("Records with AQI > 500 in raw target data:")
print(df_target[df_target["AQI"] > 500]["City"].value_counts())

print("\nMaximum raw AQI observed:", df_target["AQI"].max())

## 4. Executing the Preprocessing Pipeline
We now clean the data using `src.preprocessing.clean_and_preprocess_dataset` and verify that no nulls remain.

In [ ]:
cleaned_df = clean_and_preprocess_dataset(df_target)
save_cleaned_data(cleaned_df)

print(f"Raw records: {len(df_target)} -> Cleaned active records: {len(cleaned_df)}")
print("Total null values remaining:", cleaned_df.isna().sum().sum())
cleaned_df.describe().round(2)

## 5. Visual EDA: Distributions, Trends, and Seasonality

In [ ]:
# Figure 1: AQI Distribution Across Cities with CPCB Thresholds
city_order = cleaned_df.groupby("City")["AQI"].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 6))
sns.boxplot(data=cleaned_df, x="City", y="AQI", hue="City", order=city_order, palette="Spectral", legend=False)

plt.axhline(50, color="green", linestyle="--", label="Good (0-50)")
plt.axhline(100, color="lightgreen", linestyle="--", label="Satisfactory (51-100)")
plt.axhline(200, color="orange", linestyle="--", label="Moderate (101-200)")
plt.axhline(300, color="red", linestyle="--", label="Poor (201-300)")
plt.axhline(400, color="darkred", linestyle="--", label="Very Poor (301-400)")

plt.title("AQI Distribution Across Target Cities (CPCB Categorization)", fontsize=13, fontweight="bold")
plt.legend(loc="upper right", framealpha=0.9)
plt.show()

In [ ]:
# Figure 2: Historical AQI Trends & 2020 Lockdown Impact
plt.figure(figsize=(14, 6))
for city, color in zip(["Delhi", "Mumbai", "Bengaluru", "Kolkata"], ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]):
    c_df = cleaned_df[cleaned_df["City"] == city].sort_values("Date")
    rolling_aqi = c_df["AQI"].rolling(30, min_periods=7).mean()
    plt.plot(c_df["Date"], rolling_aqi, label=city, color=color, linewidth=2)

plt.axvspan(pd.to_datetime("2020-03-24"), pd.to_datetime("2020-05-31"), color="gold", alpha=0.3, label="COVID-19 Lockdown (2020)")
plt.title("Historical 30-Day Moving Average AQI Trajectories", fontsize=13, fontweight="bold")
plt.ylabel("AQI (30-day Rolling Mean)")
plt.legend(loc="upper right")
plt.show()

In [ ]:
# Figure 3: Monthly Seasonality (Winter Inversion vs Monsoon Washout)
cleaned_df["Month"] = cleaned_df["Date"].dt.month
monthly = cleaned_df.groupby(["City", "Month"])["AQI"].mean().reset_index()
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly, x="Month", y="AQI", hue="City", marker="o", linewidth=2)
plt.xticks(range(1, 13), month_labels)
plt.title("Monthly AQI Seasonality: Winter Smog Inversion vs. Monsoon Washout", fontsize=13, fontweight="bold")
plt.ylabel("Average AQI")
plt.show()

In [ ]:
# Figure 4: Pollutant Correlations with AQI
pollutants = ["PM2.5", "PM10", "NO2", "NOx", "CO", "SO2", "O3", "AQI"]
corr = cleaned_df[pollutants].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-0.2, vmax=1.0, square=True, linewidths=0.5)
plt.title("Correlation Matrix: Pollutants vs AQI", fontsize=13, fontweight="bold")
plt.show()

## 6. Viva Talking Points for Presentation

1. **Why PM2.5 and PM10 dominate Indian AQI:**
   - In the correlation heatmap, PM2.5 and PM10 have the strongest correlation ($r > 0.85$) with final AQI because in India's CPCB sub-index formula, the maximum sub-index usually stems from particulate matter.
2. **Meteorological Seasonality:**
   - Northern cities (Delhi, Jaipur) show extreme winter spikes (Nov-Jan) due to **thermal temperature inversion** (cold air trapped near the ground preventing pollutant dispersion) coupled with low boundary layer height and slow wind speeds.
   - Monsoon season (July-August) exhibits the cleanest air across all cities due to **wet deposition (rain washout)**.
3. **Missing Value Imputation Justification:**
   - Time-based linear interpolation was selected over mean/median imputation because air pollution is a continuous physical atmospheric process. Mean imputation creates artificial variance drops and time leakage.
4. **Outlier Treatment Rationale:**
   - Values were capped at CPCB regulatory ceiling (500) rather than deleted. Deleting rows breaks daily lag features ($t-1, t-7$) in time-series forecasting, while capping prevents gradient explosion in tree-based boosting.